<a href="https://colab.research.google.com/github/samyuktha07-prog/Story-to-Map/blob/main/storytomap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🗺️ STORY-TO-MAP
# Advanced Story Visualization Project
# Google Colab - SINGLE CELL
# No API Key Required
# ============================================================

!pip -q install -U gradio folium pandas matplotlib

import gradio as gr
import folium
import pandas as pd
import matplotlib.pyplot as plt
import re
import random
import os
import html

# ============================================================
# 🌍 LOCATION DATABASE
# ============================================================

LOCATIONS = {
    "Chennai": (13.0827, 80.2707),
    "Bangalore": (12.9716, 77.5946),
    "Bengaluru": (12.9716, 77.5946),
    "Mumbai": (19.0760, 72.8777),
    "Delhi": (28.6139, 77.2090),
    "Hyderabad": (17.3850, 78.4867),
    "Kochi": (9.9312, 76.2673),
    "Coimbatore": (11.0168, 76.9558),
    "Madurai": (9.9252, 78.1198),
    "Pondicherry": (11.9416, 79.8083),
    "Puducherry": (11.9416, 79.8083),
    "Goa": (15.2993, 74.1240),
    "Kolkata": (22.5726, 88.3639),
    "Jaipur": (26.9124, 75.7873),
    "Agra": (27.1767, 78.0081),
    "Varanasi": (25.3176, 82.9739),
    "Mysore": (12.2958, 76.6394),
    "Ooty": (11.4064, 76.6932),
    "Salem": (11.6643, 78.1460),
    "Tiruchirappalli": (10.7905, 78.7047),
    "Trichy": (10.7905, 78.7047),
    "Delhi": (28.6139, 77.2090),
    "New York": (40.7128, -74.0060),
    "London": (51.5074, -0.1278),
    "Paris": (48.8566, 2.3522),
    "Tokyo": (35.6762, 139.6503),
    "Dubai": (25.2048, 55.2708),
    "Singapore": (1.3521, 103.8198),
    "Sydney": (-33.8688, 151.2093),
    "Los Angeles": (34.0522, -118.2437),
    "California": (36.7783, -119.4179),
    "Rome": (41.9028, 12.4964),
    "Berlin": (52.5200, 13.4050),
    "Moscow": (55.7558, 37.6173),
    "Dubai": (25.2048, 55.2708),
}

# ============================================================
# 🎭 MOOD KEYWORDS
# ============================================================

MOODS = {
    "Happy": [
        "happy", "smile", "laugh", "joy", "celebrate",
        "excited", "fun", "beautiful", "wonderful"
    ],

    "Sad": [
        "sad", "cry", "tears", "alone", "lost", "pain",
        "broken", "goodbye", "miss", "grief"
    ],

    "Danger": [
        "danger", "attack", "fight", "enemy", "threat",
        "escape", "chase", "weapon", "battle"
    ],

    "Mysterious": [
        "secret", "mystery", "unknown", "shadow",
        "strange", "hidden", "clue", "dark"
    ],

    "Romantic": [
        "love", "heart", "kiss", "romantic", "together",
        "beautiful", "feelings", "forever"
    ],

    "Peaceful": [
        "calm", "peace", "quiet", "garden", "sunset",
        "morning", "nature", "relax"
    ]
}

# ============================================================
# 🧠 STORY TYPE KEYWORDS
# ============================================================

STORY_TYPES = {
    "Adventure": [
        "journey", "travel", "adventure", "mountain",
        "forest", "road", "explore", "island"
    ],

    "Mystery": [
        "mystery", "secret", "clue", "detective",
        "unknown", "investigation", "disappear"
    ],

    "Romance": [
        "love", "heart", "romance", "relationship",
        "together", "kiss", "date"
    ],

    "Thriller": [
        "danger", "chase", "murder", "enemy",
        "escape", "threat", "crime"
    ],

    "Fantasy": [
        "magic", "dragon", "kingdom", "wizard",
        "castle", "spell", "princess"
    ],

    "Drama": [
        "family", "life", "mother", "father",
        "friend", "dream", "loss", "struggle"
    ],

    "Sci-Fi": [
        "robot", "space", "planet", "future",
        "technology", "machine", "alien", "time"
    ]
}

# ============================================================
# 📍 SAMPLE STORY
# ============================================================

SAMPLE_STORY = """
Arjun lived in Chennai with his mother.

One rainy morning, he received an old letter from his father.
The letter contained a mysterious clue about a hidden treasure in Ooty.

Arjun travelled from Chennai to Ooty.

In Ooty, he met his childhood friend Maya.
Together they explored an abandoned house in the mountains.

Inside the house, they discovered another clue.
The clue pointed toward Mumbai.

Arjun and Maya travelled to Mumbai.
They found an old photograph near the Gateway of India.

Suddenly, a stranger started following them.

They escaped and travelled to Goa.
At a quiet beach, Arjun finally understood the secret behind his father's disappearance.

He returned to Chennai with Maya.
He opened the old box left by his father.

Inside was a final message:

Sometimes the greatest treasure is the journey itself.
"""

# ============================================================
# 🔎 LOCATION DETECTION
# ============================================================

def detect_locations(text):

    found = []

    for location in LOCATIONS.keys():

        pattern = r"\b" + re.escape(location) + r"\b"

        if re.search(pattern, text, re.IGNORECASE):

            if location not in found:

                found.append(location)

    return found


# ============================================================
# 🎭 MOOD DETECTION
# ============================================================

def detect_mood(text):

    words = re.findall(
        r"\b[a-zA-Z']+\b",
        text.lower()
    )

    scores = {}

    for mood, keywords in MOODS.items():

        score = 0

        for keyword in keywords:
            score += words.count(keyword)

        scores[mood] = score

    best = max(
        scores,
        key=scores.get
    )

    if scores[best] == 0:
        best = "Neutral"

    return best, scores


# ============================================================
# 📚 STORY TYPE
# ============================================================

def detect_story_type(text):

    words = re.findall(
        r"\b[a-zA-Z']+\b",
        text.lower()
    )

    scores = {}

    for story_type, keywords in STORY_TYPES.items():

        score = 0

        for keyword in keywords:
            score += words.count(keyword)

        scores[story_type] = score

    best = max(
        scores,
        key=scores.get
    )

    if scores[best] == 0:
        best = "Drama"

    return best, scores


# ============================================================
# 📖 SPLIT STORY INTO SCENES
# ============================================================

def split_story(text):

    paragraphs = re.split(
        r"\n\s*\n",
        text.strip()
    )

    paragraphs = [
        p.strip()
        for p in paragraphs
        if len(p.strip()) > 10
    ]

    if len(paragraphs) < 2:

        sentences = re.split(
            r"(?<=[.!?])\s+",
            text.strip()
        )

        sentences = [
            s.strip()
            for s in sentences
            if len(s.strip()) > 10
        ]

        paragraphs = []

        chunk = ""

        for i, sentence in enumerate(sentences):

            chunk += sentence + " "

            if (i + 1) % 3 == 0:

                paragraphs.append(
                    chunk.strip()
                )

                chunk = ""

        if chunk:
            paragraphs.append(
                chunk.strip()
            )

    return paragraphs


# ============================================================
# 🎯 FIND LOCATION FOR EACH SCENE
# ============================================================

def assign_scene_locations(scenes):

    scene_data = []

    last_location = None

    for i, scene in enumerate(scenes):

        found = detect_locations(scene)

        if found:

            location = found[0]

            last_location = location

        else:

            location = last_location

        scene_data.append({
            "Scene": i + 1,
            "Text": scene,
            "Location": location
        })

    return scene_data


# ============================================================
# 🗺️ CREATE INTERACTIVE MAP
# ============================================================

def create_map(locations, scene_data):

    if not locations:

        return None

    coordinates = [
        LOCATIONS[x]
        for x in locations
    ]

    center_lat = sum(
        x[0] for x in coordinates
    ) / len(coordinates)

    center_lon = sum(
        x[1] for x in coordinates
    ) / len(coordinates)

    story_map = folium.Map(
        location=[
            center_lat,
            center_lon
        ],
        zoom_start=5,
        tiles="CartoDB dark_matter"
    )

    # --------------------------------------------------------
    # ROUTE LINE
    # --------------------------------------------------------

    if len(coordinates) > 1:

        folium.PolyLine(
            coordinates,
            weight=5,
            opacity=0.8,
            tooltip="Story Journey"
        ).add_to(story_map)

    # --------------------------------------------------------
    # LOCATION MARKERS
    # --------------------------------------------------------

    for i, location in enumerate(locations):

        lat, lon = LOCATIONS[location]

        popup_html = f"""
        <div style="
            font-family: Arial;
            width: 220px;
        ">

        <h3>📍 Stop {i+1}: {location}</h3>

        <p>
        This location appears in the story
        journey.
        </p>

        </div>
        """

        folium.Marker(
            [lat, lon],
            popup=folium.Popup(
                popup_html,
                max_width=300
            ),
            tooltip=f"Stop {i+1}: {location}",
            icon=folium.Icon(
                color="purple",
                icon="map-marker"
            )
        ).add_to(story_map)

    # --------------------------------------------------------
    # START MARKER
    # --------------------------------------------------------

    first = locations[0]

    lat, lon = LOCATIONS[first]

    folium.Marker(
        [lat, lon],
        tooltip="🚀 Story Starts Here",
        icon=folium.Icon(
            color="green",
            icon="play"
        )
    ).add_to(story_map)

    # --------------------------------------------------------
    # END MARKER
    # --------------------------------------------------------

    if len(locations) > 1:

        last = locations[-1]

        lat, lon = LOCATIONS[last]

        folium.Marker(
            [lat, lon],
            tooltip="🏁 Story Ends Here",
            icon=folium.Icon(
                color="red",
                icon="flag"
            )
        ).add_to(story_map)

    # --------------------------------------------------------
    # SAVE MAP
    # --------------------------------------------------------

    map_path = "/content/story_map.html"

    story_map.save(map_path)

    return map_path


# ============================================================
# 📊 STORY CHART
# ============================================================

def create_chart(mood_scores, story_scores):

    moods = list(mood_scores.keys())
    mood_values = list(mood_scores.values())

    plt.figure(
        figsize=(9, 4)
    )

    plt.bar(
        moods,
        mood_values
    )

    plt.title(
        "Story Mood Analysis"
    )

    plt.xlabel(
        "Mood"
    )

    plt.ylabel(
        "Keyword Signal"
    )

    plt.xticks(
        rotation=25
    )

    plt.tight_layout()

    chart_path = "/content/story_mood_chart.png"

    plt.savefig(
        chart_path,
        dpi=150
    )

    plt.close()

    return chart_path


# ============================================================
# ⭐ STORY SCORE
# ============================================================

def calculate_story_score(
    word_count,
    location_count,
    scene_count,
    mood_count
):

    score = 0

    score += min(
        30,
        word_count / 10
    )

    score += min(
        25,
        location_count * 5
    )

    score += min(
        25,
        scene_count * 3
    )

    score += min(
        20,
        mood_count * 3
    )

    return int(
        min(100, score)
    )


# ============================================================
# 🎬 STORY TITLE
# ============================================================

def generate_title(story_type):

    titles = {

        "Adventure": [
            "The Journey Beyond",
            "Roads of Destiny",
            "The Lost Route",
            "Beyond the Horizon"
        ],

        "Mystery": [
            "The Hidden Map",
            "The Last Clue",
            "Secrets of the Journey",
            "The Missing Chapter"
        ],

        "Romance": [
            "Where Our Roads Meet",
            "Two Hearts, One Journey",
            "Across the Distance",
            "The Road to You"
        ],

        "Thriller": [
            "The Final Route",
            "Chasing the Shadow",
            "No Way Back",
            "The Escape"
        ],

        "Fantasy": [
            "The Forgotten Kingdom",
            "Beyond the Magic Road",
            "The Hidden Realm",
            "Journey to Tomorrow"
        ],

        "Drama": [
            "The Journey Within",
            "Roads We Remember",
            "A Thousand Miles",
            "The Last Journey"
        ],

        "Sci-Fi": [
            "Destination 2099",
            "The Future Route",
            "Beyond Time",
            "The Last Transmission"
        ]
    }

    return random.choice(
        titles.get(
            story_type,
            titles["Drama"]
        )
    )


# ============================================================
# 🧠 MAIN ANALYZER
# ============================================================

def analyze_story(story):

    if not story or len(story.strip()) < 30:

        return (
            "⚠️ Please enter a longer story.",
            "",
            "",
            "",
            None,
            None
        )

    # --------------------------------------------------------
    # BASIC INFORMATION
    # --------------------------------------------------------

    words = re.findall(
        r"\b[a-zA-Z']+\b",
        story
    )

    word_count = len(words)

    sentences = re.split(
        r"[.!?]+",
        story
    )

    sentences = [
        s.strip()
        for s in sentences
        if s.strip()
    ]

    sentence_count = len(
        sentences
    )

    # --------------------------------------------------------
    # DETECTION
    # --------------------------------------------------------

    locations = detect_locations(
        story
    )

    story_type, story_scores = detect_story_type(
        story
    )

    mood, mood_scores = detect_mood(
        story
    )

    scenes = split_story(
        story
    )

    scene_data = assign_scene_locations(
        scenes
    )

    story_score = calculate_story_score(
        word_count,
        len(locations),
        len(scenes),
        len([x for x in mood_scores.values() if x > 0])
    )

    title = generate_title(
        story_type
    )

    # --------------------------------------------------------
    # MAP
    # --------------------------------------------------------

    map_path = create_map(
        locations,
        scene_data
    )

    # --------------------------------------------------------
    # CHART
    # --------------------------------------------------------

    chart_path = create_chart(
        mood_scores,
        story_scores
    )

    # ========================================================
    # DASHBOARD
    # ========================================================

    dashboard = f"""
# 🗺️ {title}

## ✨ STORY JOURNEY DETECTED

### ⭐ Story Exploration Score: **{story_score}/100**

---

| 📊 Story Element | Result |
|---|---|
| 📚 Story Type | **{story_type}** |
| ❤️ Dominant Mood | **{mood}** |
| 🗺️ Locations Found | **{len(locations)}** |
| 🎬 Scenes | **{len(scenes)}** |
| 📝 Words | **{word_count}** |
| 📄 Sentences | **{sentence_count}** |

---

## 📍 JOURNEY

"""

    if locations:

        for i, location in enumerate(
            locations,
            1
        ):

            emoji = "🚀" if i == 1 else (
                "🏁" if i == len(locations)
                else "📍"
            )

            dashboard += f"""
### {emoji} Stop {i}: {location}
"""

    else:

        dashboard += """
### ❌ No known locations detected

Try mentioning cities or places such as:

**Chennai, Mumbai, Ooty, Goa, London, Tokyo**
"""

    # ========================================================
    # STORY TYPE ANALYSIS
    # ========================================================

    intelligence = f"""
# 🧠 STORY INTELLIGENCE

## 🎭 Story Type

**{story_type}**

### Genre Signals

"""

    sorted_story = sorted(
        story_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    for name, value in sorted_story:

        intelligence += f"""
- **{name}** → {value}
"""

    intelligence += """

---

## ❤️ Emotional Landscape

"""

    sorted_moods = sorted(
        mood_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    for name, value in sorted_moods:

        intelligence += f"""
- **{name}** → {value}
"""

    intelligence += f"""

---

## 🌍 Geographic Spread

Your story travels through:

**{len(locations)} known location(s)**

"""

    if len(locations) >= 4:

        intelligence += """
🌎 **Highly mobile story**

Your characters travel through multiple locations,
making this suitable for an adventure or road-story structure.
"""

    elif len(locations) >= 2:

        intelligence += """
🚗 **Multi-location story**

The narrative changes locations enough to create
a clear sense of movement.
"""

    elif len(locations) == 1:

        intelligence += """
🏠 **Single-location story**

The story mainly revolves around one place.
"""

    else:

        intelligence += """
📖 **Location-light story**

The story focuses more on events and characters
than geography.
"""

    # ========================================================
    # SCENE TIMELINE
    # ========================================================

    timeline = """
# 🎬 STORY TIMELINE

"""

    for i, item in enumerate(
        scene_data,
        1
    ):

        text = item["Text"]

        short_text = (
            text[:250] + "..."
            if len(text) > 250
            else text
        )

        location = item["Location"]

        scene_mood, _ = detect_mood(
            text
        )

        timeline += f"""
### 🎞️ Scene {i}

**📍 Location:** {location if location else "Unknown"}

**❤️ Mood:** {scene_mood}

**📝 Event:**

{short_text}

---

"""

    # ========================================================
    # DOWNLOAD REPORT
    # ========================================================

    report = f"""
=========================================================
                 STORY-TO-MAP REPORT
=========================================================

Generated Title:
{title}

Story Type:
{story_type}

Dominant Mood:
{mood}

Story Exploration Score:
{story_score}/100

---------------------------------------------------------
STORY STATISTICS
---------------------------------------------------------

Words:
{word_count}

Sentences:
{sentence_count}

Scenes:
{len(scenes)}

Locations:
{len(locations)}

---------------------------------------------------------
JOURNEY
---------------------------------------------------------

"""

    for i, location in enumerate(
        locations,
        1
    ):

        report += f"{i}. {location}\n"

    report += """
---------------------------------------------------------
SCENE TIMELINE
---------------------------------------------------------

"""

    for item in scene_data:

        report += f"""
Scene {item["Scene"]}
Location: {item["Location"] or "Unknown"}
Text: {item["Text"][:300]}

"""

    report += """
=========================================================
Generated by Story-to-Map
=========================================================
"""

    report_path = "/content/story_to_map_report.txt"

    with open(
        report_path,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(report)

    return (
        dashboard,
        intelligence,
        timeline,
        report,
        map_path,
        chart_path
    )


# ============================================================
# 🧪 SAMPLE BUTTON
# ============================================================

def load_sample():

    return SAMPLE_STORY


# ============================================================
# 🎨 CUSTOM CSS
# ============================================================

css = """

.gradio-container {

    max-width: 1250px !important;

    margin: auto !important;

}

#hero {

    text-align: center;

    padding: 30px;

    border-radius: 24px;

    margin-bottom: 25px;

    background:
    linear-gradient(
        135deg,
        #151515,
        #282828
    );

    border: 1px solid #444;

}

#hero h1 {

    font-size: 44px;

    margin-bottom: 8px;

}

#hero p {

    font-size: 17px;

    opacity: 0.85;

}

textarea {

    font-size: 15px !important;

}

button {

    border-radius: 12px !important;

}

"""

# ============================================================
# 🖥️ GRADIO APP
# ============================================================

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="purple",
        secondary_hue="pink"
    ),
    css=css,
    title="Story-to-Map"
) as app:

    # --------------------------------------------------------
    # HERO
    # --------------------------------------------------------

    gr.HTML("""
    <div id="hero">

        <h1>🗺️ STORY → MAP</h1>

        <p>
        Transform a story into an interactive geographic journey.
        </p>

        <p>
        📍 Locations • 🎬 Scenes • ❤️ Mood • 🧠 Story Type • 🌍 Journey
        </p>

    </div>
    """)

    # --------------------------------------------------------
    # INPUT AREA
    # --------------------------------------------------------

    with gr.Row():

        with gr.Column(
            scale=1
        ):

            gr.Markdown(
                "## ✍️ WRITE YOUR STORY"
            )

            story_input = gr.Textbox(

                label="Story Input",

                placeholder="""
Example:

Arun lived in Chennai.

He travelled to Ooty to find a hidden secret.

Later, he went to Mumbai...

""",

                lines=22

            )

            with gr.Row():

                analyze_btn = gr.Button(
                    "🗺️ CREATE STORY MAP",
                    variant="primary"
                )

                sample_btn = gr.Button(
                    "✨ LOAD SAMPLE"
                )

                clear_btn = gr.Button(
                    "🗑️ CLEAR"
                )

        with gr.Column(
            scale=1
        ):

            gr.Markdown(
                "## 📊 STORY DASHBOARD"
            )

            dashboard_output = gr.Markdown()

    # ========================================================
    # TABS
    # ========================================================

    with gr.Tabs():

        # ----------------------------------------------------
        # MAP
        # ----------------------------------------------------

        with gr.Tab(
            "🗺️ Interactive Map"
        ):

            map_output = gr.HTML(
                label="Story Map"
            )

        # ----------------------------------------------------
        # INTELLIGENCE
        # ----------------------------------------------------

        with gr.Tab(
            "🧠 Story Intelligence"
        ):

            intelligence_output = gr.Markdown()

        # ----------------------------------------------------
        # TIMELINE
        # ----------------------------------------------------

        with gr.Tab(
            "🎬 Story Timeline"
        ):

            timeline_output = gr.Markdown()

        # ----------------------------------------------------
        # REPORT
        # ----------------------------------------------------

        with gr.Tab(
            "📄 Report"
        ):

            report_output = gr.Textbox(

                label="Generated Story Report",

                lines=20,

                interactive=False

            )

        # ----------------------------------------------------
        # CHART
        # ----------------------------------------------------

        with gr.Tab(
            "📊 Mood Chart"
        ):

            chart_output = gr.Image(
                label="Story Mood"
            )

    # ========================================================
    # MAP HTML LOADER
    # ========================================================

    def load_map_html(path):

        if not path:
            return """
            <div style="
                padding:40px;
                text-align:center;
            ">
            🗺️ No map available.
            </div>
            """

        try:

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as file:

                content = file.read()

            return content

        except:

            return """
            <div style="
                padding:40px;
                text-align:center;
            ">
            ⚠️ Unable to load map.
            </div>
            """

    # ========================================================
    # ANALYZE EVENT
    # ========================================================

    def full_analysis(story):

        result = analyze_story(
            story
        )

        dashboard = result[0]
        intelligence = result[1]
        timeline = result[2]
        report = result[3]
        map_path = result[4]
        chart_path = result[5]

        map_html = load_map_html(
            map_path
        )

        return (
            dashboard,
            intelligence,
            timeline,
            report,
            map_html,
            chart_path
        )

    analyze_btn.click(

        fn=full_analysis,

        inputs=story_input,

        outputs=[
            dashboard_output,
            intelligence_output,
            timeline_output,
            report_output,
            map_output,
            chart_output
        ]

    )

    # ========================================================
    # SAMPLE
    # ========================================================

    sample_btn.click(

        fn=load_sample,

        outputs=story_input

    )

    # ========================================================
    # CLEAR
    # ========================================================

    clear_btn.click(

        fn=lambda: "",

        outputs=story_input

    )


# ============================================================
# 🚀 LAUNCH
# ============================================================

app.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_1634/1057774146.py:1084: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d529c5fb287d002f9d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
/usr/local/lib/python3.13/dist-packages/gradio/components/html.py:208: UserWarning: A `<script>` tag was found in the content of a `gr.HTML` component. Browsers do not execute `<script>` tags inserted via `innerHTML`, so this script will not run. Use the `head` parameter to load external libraries (e.g. `head='<script src="..."></script>'`); `head` content is injected and loaded before `js_on_load` runs. Then, if needed, put code that uses those libraries in the `js_on_load` parameter, which executes when the component renders. See https://gradio.app/guides/custom-HTML-components for details.
  _warn_if_script_tag(value)
